# 02 · The RAG pipeline, one stage at a time

**Objective.** Understand - and debug - every stage between a question and the answer. Each stage calls the same code
the chatbot uses (`vitala_rag.retrieval`, `vitala_rag.chatbot`), so what you see here is what production does.

1. Load data · 2. Clean and chunk documents · 3. Generate embeddings · 4. Build retrieval indexes ·
5. Process user questions · 6. Retrieve and rank evidence · 7. Generate answers · 8. Validate citations and grounding ·
9. Return patient-friendly responses

`USE_LLM = False` (below) runs everything without an API key: the answer step then quotes evidence verbatim.

In [ ]:
# Stage 0 - setup: find the project root, make `src/` importable, load .env (keys are never printed)
import sys, os, json
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "src" / "vitala_rag").exists():
    if ROOT.parent == ROOT:
        raise RuntimeError("Open this notebook from inside the VITALA_RAG project (could not find src/vitala_rag).")
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
os.chdir(ROOT)
try:
    from dotenv import load_dotenv
    load_dotenv(ROOT / ".env", override=False)
except ImportError:
    pass

import pandas as pd
pd.set_option("display.max_colwidth", 120)
from vitala_rag.config import Paths
paths = Paths(root=ROOT)
print("Project root:", ROOT)
print("LLM provider configured:", os.environ.get("VITALA_LLM_PROVIDER", "(not set)"), "| API keys are not displayed")

In [ ]:
USE_LLM = bool(os.environ.get("AZURE_OPENAI_API_KEY") or os.environ.get("ANTHROPIC_API_KEY"))   # set False to stay offline
PROVIDER = os.environ.get("VITALA_LLM_PROVIDER", "extractive") if USE_LLM else "extractive"
QUESTION = "Can ACE inhibitors cause a dry cough?"
print("provider:", PROVIDER)

## 1 · Load data

Three on-disk collections (FAISS + BM25 each) and the synthetic patient store, which is kept separate from the medical
knowledge. `HybridRetriever` holds the two medical collections; `AppKnowledgeBase` the app documentation.

In [ ]:
from vitala_rag.retrieval import Embedder, HybridRetriever, AppKnowledgeBase, verify_index
embedder = Embedder()                       # multilingual-e5-base; first run downloads the model
retriever = HybridRetriever(paths.indexes, embedder=embedder)
app_kb = AppKnowledgeBase(paths, embedder=embedder)
print({c: len(ix.chunks) for c, ix in retriever.indexes.items()}, "| app:", len(app_kb.chunks), "-", app_kb.index_note)

## 2 · Clean and chunk documents

Chunking happened in notebook 01. Here: how big the chunks are per source. Guideline chunks target ~1,400 characters;
Saudi KB records are naturally short (one record each); tables are separate chunks.

In [ ]:
chunks = pd.DataFrame([c for ix in retriever.indexes.values() for c in ix.chunks])
chunks.groupby("source_id")["char_len"].describe()[["count", "mean", "min", "max"]].round(0)

## 3 · Generate embeddings

e5 needs the prefixes `query: ` / `passage: `. Vectors are L2-normalised, so the inner product **is** the cosine
similarity. Arabic and English questions with the same meaning land close together - that is what makes cross-lingual
retrieval work against English sources.

In [ ]:
import numpy as np
qs = ["Can ACE inhibitors cause a dry cough?", "هل يسبب الليسينوبريل كحة؟", "How do I book an appointment?"]
v = embedder.encode_queries(qs)
print("dimension:", v.shape[1])
pd.DataFrame(np.round(v @ v.T, 3), index=qs, columns=qs)

## 4 · Build retrieval indexes

Each collection has `faiss.index` (dense), `bm25_tokens.json` (keyword; BM25 is rebuilt in memory at load) and
`index_manifest.json` (model, counts and hashes). The app collection's BM25 uses Arabic character 3-grams, because Saudi
dialect changes word shapes (احجز / الحجز / حجز).

In [ ]:
pd.DataFrame([verify_index(paths.indexes / c) for c in ("preventive", "medication", "app")])

## 5 · Process user questions

Before retrieval: **safety rules** (emergencies, dose-change requests, diagnosis requests, prompt injection), **language**,
an Arabic->English **keyword expansion** for BM25, **medication scope** (metformin + ACE inhibitors only) and **routing**
to a collection.

In [ ]:
from vitala_rag.retrieval import process_query
from vitala_rag.safety import check_question
for q in [QUESTION, "هل يسبب الليسينوبريل كحة؟", "What are the side effects of amlodipine?", "كيف احجز موعد؟",
          "My lips are swelling after my new lisinopril"]:
    pq, s = process_query(q), check_question(q)
    print(f"{q[:45]:47s} lang={pq.language} collections={pq.collections} drugs={pq.drugs} other={pq.other_drugs} "
          f"app={pq.app_intent} emergency={s.emergency} flags={s.flags}")

## 6 · Retrieve and rank evidence

Dense (FAISS) and BM25 rankings are fused with **Reciprocal Rank Fusion**: `score = Σ 1/(60 + rank)`. RRF needs no score
normalisation, which matters because cosine and BM25 scores live on different scales. The **evidence gate** then refuses
to answer when the best dense similarity is below `MIN_DENSE_SCORE` (0.82, calibrated on the dev split).

`retrieve_only()` runs routing + retrieval without any LLM call: if the right chunk is not in this list, the problem is
retrieval, not generation.

In [ ]:
from vitala_rag.chatbot import VitalaRAG, get_llm
rag = VitalaRAG(paths=paths, llm=get_llm(PROVIDER), embedder=embedder, translate_queries=USE_LLM, app_kb=app_kb, log=False)
r = rag.retrieve_only(QUESTION)
print("route:", r["route"], "| collections:", r["collections"], "| medications:", r["medications"])
print("gate:", r["evidence_gate"], "| timings (ms):", r["timings_ms"])
pd.DataFrame(r["candidates"])[["rank", "chunk_id", "dense_score", "dense_rank", "bm25_score", "bm25_rank", "rrf", "section"]]

## 7 · Generate answers

The prompt holds the system rules (cite every sentence, use only the evidence, plain language, no dosing advice...) and
the evidence passages, each labelled with its source type. Look at exactly what the LLM receives:

In [ ]:
from vitala_rag.chatbot import SYSTEM_PROMPT, build_user_prompt
pq = process_query(QUESTION)
cands = rag.retriever.search(pq)
flags = []
evidence = rag._evidence(cands, flags)          # sanitised evidence with [E#] ids
prompt = build_user_prompt(QUESTION, pq.language, evidence, None)
print(prompt[:2500], "\n...")

In [ ]:
answer = rag.llm.generate(SYSTEM_PROMPT, prompt)
print(answer)

## 8 · Validate citations and grounding

RAG does not prevent hallucination by itself, so every answer is checked: valid `[E#]` markers, a citation on every
factual sentence, every number present in the cited evidence, and each sentence supported by its cited passage (word
overlap, or embedding similarity for cross-lingual answers). A failed check triggers one retry with feedback, then a
verbatim-quote fallback.

In [ ]:
from vitala_rag.chatbot import check_grounding
rep = check_grounding(answer, evidence, embedder)
print("verdict:", rep.verdict, rep.reasons)
pd.DataFrame(rep.sentence_details)[["sentence", "citations", "supported"] + [c for c in ("lexical_recall", "embed_sim", "why")
                                                                         if c in pd.DataFrame(rep.sentence_details)]]

## 9 · Return patient-friendly responses

`rag.answer()` runs all stages and returns the API response. `debug=True` adds a **developer-only** trace (never shown to
patients; the API returns it only when the server runs with `VITALA_DEBUG=1`).

In [ ]:
resp = rag.answer(QUESTION, debug=True)
print(resp["status"], "| route:", resp["route"], "| timings (ms):", resp["timings_ms"], "| LLM:", resp["llm_usage"])
print("\n" + resp["answer"])
for c in resp["citations"]:
    print(f"  [{c['evidence_id']}] {c['citation']}")

In [ ]:
from vitala_rag.evaluation import readability
dbg = resp["debug"]
print("debug keys:", list(dbg))
print("readability proxy:", readability(resp["answer"], resp["language"]))
pd.DataFrame(dbg.get("llm_calls", []))[[c for c in ("purpose", "latency_ms", "input_tokens", "output_tokens") if dbg.get("llm_calls")]]

**Expected result.** Status `answered`, citations from the Saudi medication knowledge base and/or StatPearls, a
grounding verdict of `pass`, and an explanation in plain words. With `USE_LLM = False` the answer is a list of verbatim
quotes instead.